<p align="center"><img src="https://raw.githubusercontent.com/arab-future-academy/deep_dive_in_gen_ai/refs/heads/main/imgs/arabfutureacademy.png" alt="Arabic Future Academy" width="720" /></p>

# ComfyUI on Kaggle


In [ ]:
%%time
update = False

import os
import socket
import stat
import subprocess
#!rm -rf /kaggle/working/venv
home_dir = '/kaggle/working'
python = '/kaggle/working/venv/bin/python'
pip = '/kaggle/working/venv/bin/pip'

REQUIRED_HOSTS = ['pypi.org', 'github.com', 'raw.githubusercontent.com']

def check_internet_or_stop():
    failed = []
    for host in REQUIRED_HOSTS:
        try:
            socket.gethostbyname(host)
        except OSError as err:
            failed.append(f'{host}: {err}')
    if failed:
        raise RuntimeError(
            'Kaggle internet/DNS is not working, so setup cannot continue. '
            'In Kaggle, open Notebook settings and turn Internet on, then restart the session and run this cell again. '
            'Failed DNS checks: ' + '; '.join(failed)
        )
    print('[OK] Internet/DNS checks passed.')

check_internet_or_stop()

def check_gpu_or_stop():
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=False)
    except FileNotFoundError:
        result = None
    if result is None or result.returncode != 0:
        raise RuntimeError(
            'No NVIDIA GPU driver was detected. In Kaggle, open Session options, set Accelerator to GPU T4 x2 or GPU T4, '
            'then restart the session and run setup again. ComfyUI cannot use CUDA in a CPU-only session.'
        )
    print('[OK] NVIDIA GPU driver detected.')
    print(result.stdout.splitlines()[0] if result.stdout else 'nvidia-smi passed')

check_gpu_or_stop()

def find_bin_folders(folder_path):
    bin_folders = []
    for root, dirs, files in os.walk(folder_path):
        for dir_name in dirs:
            if dir_name == 'bin':
                bin_folders.append(os.path.join(root, dir_name)) 
    return bin_folders

def installLibraries(home_dir, python, pip):
  %cd {home_dir}
  !{pip} install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
  # TensorFlow is intentionally not installed. ComfyUI uses PyTorch, and TensorFlow CUDA packages
  # can replace PyTorch CUDA 12.1 NVIDIA wheels with incompatible newer CUDA wheels on Kaggle.
  # Extra remote req.txt install removed: ComfyUI requirements are installed from the cloned repo below.

!pip install virtualenv

if not os.path.exists(f'{home_dir}/venv'):
    print('installing venv')
    os.chdir(home_dir)
    get_ipython().system(f'cd {home_dir}')
    
    get_ipython().system('virtualenv venv -p $(which python3.10)')
    installLibraries(home_dir, python, pip)
else:
    bin_folders = find_bin_folders('/kaggle/working/venv')
    if bin_folders:
      print("Found 'bin' folders:")
      for bin_folder in bin_folders:
        print(bin_folder)
        for filename in os.listdir(bin_folder):
            file_path = os.path.join(bin_folder, filename)
            if os.path.isfile(file_path):
                current_permissions = os.stat(file_path).st_mode
                 # Add execute permissions for the user, group, and others
                os.chmod(file_path, current_permissions | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

# Keep virtualenv's Python launchers as created. Forcing python/python3 symlinks can create loops on Kaggle.

%cd /kaggle/working
if not os.path.exists('/kaggle/working/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI

!{pip} install -r requirements.txt

!mkdir -p /tmp/models/checkpoints
!mkdir -p /tmp/models/clip
!mkdir -p /tmp/models/vae
!mkdir -p /tmp/models/unet

# Remove the following lines to keep models in permanent storage.
!rm -rf /kaggle/working/ComfyUI/models/checkpoints
!rm -rf /kaggle/working/ComfyUI/models/clip
!rm -rf /kaggle/working/ComfyUI/models/vae
!rm -rf /kaggle/working/ComfyUI/models/unet

!ln -s /tmp/models/checkpoints /kaggle/working/ComfyUI/models/checkpoints
!ln -s /tmp/models/clip /kaggle/working/ComfyUI/models/clip
!ln -s /tmp/models/vae /kaggle/working/ComfyUI/models/vae
!ln -s /tmp/models/unet /kaggle/working/ComfyUI/models/unet

checkpoints =  '/kaggle/working/ComfyUI/models/checkpoints'
link_path = checkpoints + '/temp-models'
temp_models = '/kaggle/temp/temp-models'

!mkdir -p /kaggle/temp
!mkdir -p $temp_models

if not os.path.exists(link_path):
    get_ipython().system(f'ln -s {temp_models} {checkpoints}')

# Install the node manager
update_manager = True
%cd /kaggle/working/ComfyUI/custom_nodes
if not os.path.exists('/kaggle/working/ComfyUI/custom_nodes/ComfyUI-Manager'):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git
%cd ComfyUI-Manager
!git pull

if update_manager:
    get_ipython().system('git pull')
!{pip} install -r requirements.txt

# Second GPU offload
%cd /kaggle/working/ComfyUI/custom_nodes
!wget https://gist.githubusercontent.com/city96/30743dfdfe129b331b5676a79c3a8a39/raw/ecb4f6f5202c20ea723186c93da308212ba04cfb/ComfyBootlegOffload.py


---
# Model Management

## Install a model

Copy the model URL to the model_url field. Make sure the model can be accessed publicly, without being signed into a website.

In [ ]:
#### Install a check point in permanent storage
# Make sure Persistence is set to "Files only" or "Variables and Files"
model_url = 'https://huggingface.co/Lightricks/LTX-2.3-fp8/resolve/main/ltx-2.3-22b-dev-fp8.safetensors?download=true'
model_name = 'ltx-2.3-22b-dev-fp8.safetensors'
%cd $checkpoints
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

In [ ]:
# Install a LoRA in permanent storage
model_url = 'htt /kaggle/working/ComfyUI/models/lorasps://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-22b-distilled-lora-384.safetensors?download=true'
model_name = 'ltx-2.3-22b-distilled-lora-384.safetensors'

%cd
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

In [ ]:
# Install a text encoder in permanent storage
model_url = 'https://huggingface.co/Comfy-Org/ltx-2/resolve/main/split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors?download=true'
model_name = 'gemma_3_12B_it_fp4_mixed.safetensors'
text_encoders
text_encoders = '/kaggle/working/ComfyUI/models/text_encoders'
!mkdir -p $
%cd $text_encoders
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

In [ ]:
# Install a latent upscale model in permanent storage
model_url = 'https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-spatial-upscaler-x2-1.1.safetensors?download=true'
model_name = 'ltx-2.3-spatial-upscaler-x2-1.1.safetensors'

latent_upscale_models = '/kaggle/working/ComfyUI/models/latent_upscale_models'
!mkdir -p $latent_upscale_models
%cd $latent_upscale_models
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

--- 
# WebUI

## Start the WebUI with Cloudflare Tunnel
* Wait for a `trycloudflare.com` URL, then open it.
* <span style="color: red; font-weight: 700;">Keep this cell running while using ComfyUI.</span>

In [ ]:
# Start the WebUI with Cloudflare Tunnel
# This starts ComfyUI in the background and keeps cloudflared in the foreground so Kaggle shows the public URL.

import os
import subprocess
import time

COMFYUI_DIR = '/kaggle/working/ComfyUI'
VENV_DIR = '/kaggle/working/venv'
PORT = 8188
CLOUDFLARED = '/kaggle/working/cloudflared'

python_candidates = [
    f'{VENV_DIR}/bin/python',
    f'{VENV_DIR}/bin/python3',
    f'{VENV_DIR}/bin/python3.10',
]
PYTHON = next((candidate for candidate in python_candidates if os.path.exists(candidate)), None)
if PYTHON is None:
    raise FileNotFoundError(
        'No Python executable was found in /kaggle/working/venv/bin. Run the installation/setup cell first. '
        'If the venv was damaged, run: !rm -rf /kaggle/working/venv, then rerun the setup cell.'
    )

if not os.path.exists(f'{COMFYUI_DIR}/main.py'):
    raise FileNotFoundError('ComfyUI was not found. Run the installation/setup cell first.')

%cd {COMFYUI_DIR}

if not os.path.exists(CLOUDFLARED):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {CLOUDFLARED}
    !chmod a+x {CLOUDFLARED}

print('Using Python:', PYTHON)
print('Starting ComfyUI in the background...')
comfy_log = '/kaggle/working/comfyui_cloudflared.log'
comfy_cmd = [PYTHON, f'{COMFYUI_DIR}/main.py', '--listen', '127.0.0.1', '--port', str(PORT)]
with open(comfy_log, 'w') as log:
    subprocess.Popen(comfy_cmd, cwd=COMFYUI_DIR, stdout=log, stderr=subprocess.STDOUT)

time.sleep(8)
print('If ComfyUI fails to start, check:', comfy_log)
print('Starting Cloudflare Tunnel. Open the trycloudflare.com URL printed below.')
!{CLOUDFLARED} tunnel --url http://127.0.0.1:{PORT}
